# 🤖 Suraksha - Two-Stage Model Training (Baseline Solution)

## Objective
Train a **two-stage fraud detection system** using **real fraud labels** + **pattern-based explanations**.

## Two-Stage Approach (Best of Both Worlds)

### Stage 1: Fraud Detection (Binary Classifier)
* **Input**: Pattern features from `workspace.silver.upi_pattern_features`
* **Target**: **REAL fraud labels** (`is_fraud`: 480 fraud cases, 0.19%)
* **Purpose**: Detect whether transaction is fraud or legitimate
* **Output**: Binary prediction (0 = Legitimate, 1 = Fraud)

### Stage 2: Fraud Type Classification (Multiclass Classifier)
* **Input**: Same pattern features
* **Target**: Pattern-based type labels (`pattern_fraud_type_id`: 0-4)
* **Purpose**: For detected fraud, explain WHAT TYPE of suspicious pattern it exhibits
* **Output**: Pattern type (Amount Anomaly, Temporal Anomaly, Merchant Fraud, High-Risk Pattern)

## Why Two-Stage?

✅ **Solves Data Leakage**: Stage 1 uses REAL fraud labels (not derived from features)  
✅ **Demonstrates Features**: Stage 2 shows pattern features work for classification  
✅ **Realistic Metrics**: Stage 1 accuracy reflects real-world fraud detection challenge  
✅ **Actionable Insights**: Stage 2 provides explanation for WHY fraud was flagged  

## Inference Flow
```
Transaction → [Stage 1: Binary] → Is Fraud?
                                    ↓ YES
                            [Stage 2: Multiclass] → Fraud Type?
                                    ↓
                        "Temporal Anomaly" (2-5 AM pattern)
                                    ↓
                            [RAG Pipeline] → RBI Guidelines
                                    ↓
                            User sees explanation
```

## Pattern-Based Fraud Types (Stage 2)
1. **Legitimate** (0) - No suspicious patterns
2. **Amount Anomaly** (1) - Statistical outliers (z-score > 3)
3. **Temporal Anomaly** (2) - Odd hours (2-5 AM, late night)
4. **Merchant Fraud** (3) - High-risk merchant categories
5. **High-Risk Pattern** (4) - Multiple indicators (risk score ≥ 8)

## Key Differences from Advanced Solution
| Feature | Baseline (Two-Stage) | Advanced |
|---------|----------|----------|
| **Stage 1 Target** | is_fraud (real, 0.19%) | Multiclass (synthetic, 5%) |
| **Stage 2 Purpose** | Explain pattern type | N/A (single-stage multiclass) |
| **User Tracking** | ❌ None | ✓ VPAs, Device IDs |
| **Fraud Types** | 4 pattern-based | 9 (5 behavior + 4 pattern) |
| **Data Source** | Official dataset | Synthetic with user IDs |
| **Approach** | Detection + Explanation | Direct multiclass prediction |

In [0]:
%pip install xgboost shap

In [0]:
# Import required libraries
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    precision_recall_curve, roc_curve, f1_score,
    accuracy_score, precision_score, recall_score
)
import xgboost as xgb

# MLflow for experiment tracking
import mlflow
import mlflow.xgboost
import mlflow.sklearn

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# SHAP for explainability
import shap

import warnings
warnings.filterwarnings('ignore')

# Get dynamic username for reproducibility
username = spark.sql("SELECT current_user()").collect()[0][0]

# Configuration
source_table = "workspace.silver.upi_pattern_features"
model_name = "suraksha_baseline"
experiment_name = f"/Users/{username}/suraksha_baseline_experiment"

print("✓ Libraries imported successfully")
print(f"Username: {username}")
print(f"Source: {source_table}")
print(f"Model name: {model_name}")
print(f"Experiment: {experiment_name}")

In [0]:
# Load pattern features from silver layer
print("Loading pattern features from silver layer...")
df_silver = spark.read.table(source_table)

print(f"✓ Loaded {df_silver.count():,} transactions")
print(f"\nSchema:")
df_silver.printSchema()

# Show fraud distribution
print("\nFraud distribution:")
fraud_dist = df_silver.groupBy("is_pattern_fraud").count().toPandas()
print(fraud_dist)

# Show pattern fraud types
print("\nPattern fraud type distribution:")
pattern_dist = df_silver.groupBy("pattern_fraud_type").count().orderBy("count", ascending=False).toPandas()
print(pattern_dist)

In [0]:
# Define feature columns (pattern-based only, no user tracking)
feature_cols = [
    # Temporal features
    'hour', 'day_of_week', 'month',
    'is_odd_hours', 'is_weekend', 'is_late_night', 'is_business_hours',
    
    # Amount features
    'amount_inr', 'amount_zscore',
    'is_high_amount', 'is_very_high_amount', 'is_round_amount',
    
    # Merchant features
    'is_high_risk_merchant', 'is_medium_risk_merchant', 
    'is_p2m', 'merchant_risk_score',
    
    # Transaction patterns
    'failed_txn', 'large_round_amount',
    'odd_hour_high_amount', 'weekend_high_amount',
    'high_risk_merchant_high_amount',
    
    # Risk scores
    'temporal_risk', 'amount_risk', 'pattern_risk', 'total_risk_score'
]

# TWO-STAGE APPROACH:
# Stage 1: Binary classifier on REAL fraud labels (is_fraud)
# Stage 2: Multiclass classifier on pattern types (pattern_fraud_type_id) for explanation
target_col_stage1 = 'is_fraud'  # Real fraud labels (480 cases, 0.19%)
target_col_stage2 = 'pattern_fraud_type_id'  # Pattern types (0-4) for explanation

print(f"Feature columns selected: {len(feature_cols)}")
print(f"\nStage 1 Target: {target_col_stage1} (REAL fraud detection)")
print(f"Stage 2 Target: {target_col_stage2} (Pattern type classification)")
print(f"\nFeatures:")
for i, col_name in enumerate(feature_cols, 1):
    print(f"  {i}. {col_name}")

# Select features and both targets
df_ml = df_silver.select(feature_cols + [target_col_stage1, target_col_stage2])

print(f"\n✓ Selected {len(feature_cols)} features for training")
print(f"✓ Two targets: fraud detection (real) + pattern classification")

In [0]:
# Convert to Pandas for sklearn/xgboost
print("Converting to Pandas DataFrame...")
df_pandas = df_ml.toPandas()

print(f"✓ Converted to Pandas: {len(df_pandas):,} rows, {len(df_pandas.columns)} columns")

# Check for missing values
print("\nChecking for missing values...")
missing_counts = df_pandas.isnull().sum()
if missing_counts.sum() > 0:
    print("Missing values found:")
    print(missing_counts[missing_counts > 0])
    print("\nFilling missing values with 0...")
    df_pandas = df_pandas.fillna(0)
    print("✓ Missing values handled")
else:
    print("✓ No missing values found")

# Convert boolean columns to int
print("\nConverting boolean columns to integers...")
bool_cols = df_pandas.select_dtypes(include=['bool']).columns.tolist()
if bool_cols:
    print(f"Boolean columns: {bool_cols}")
    df_pandas[bool_cols] = df_pandas[bool_cols].astype(int)
    print("✓ Boolean columns converted")

# Display data types
print("\nData types:")
print(df_pandas.dtypes)

# Display basic statistics
print("\nBasic statistics:")
print(df_pandas.describe())

In [0]:
# Separate features and targets (both stages)
X = df_pandas[feature_cols]
y_stage1 = df_pandas[target_col_stage1]  # is_fraud (binary)
y_stage2 = df_pandas[target_col_stage2]  # pattern_fraud_type_id (multiclass 0-4)

print(f"Feature matrix shape: {X.shape}")
print(f"\nStage 1 Target (is_fraud):")
print(y_stage1.value_counts().sort_index())
print(f"Fraud rate: {(y_stage1.sum() / len(y_stage1) * 100):.2f}%")

print(f"\nStage 2 Target (pattern_fraud_type_id):")
print(y_stage2.value_counts().sort_index())

# Split into train and test sets (80-20 split)
# Use stratify on Stage 1 target (real fraud labels)
X_train, X_test, y_train_s1, y_test_s1, y_train_s2, y_test_s2 = train_test_split(
    X, y_stage1, y_stage2,
    test_size=0.2, 
    random_state=42, 
    stratify=y_stage1  # Maintain fraud class balance
)

print(f"\n✓ Data split completed")
print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")
print(f"\nTraining set fraud rate (Stage 1): {(y_train_s1.sum() / len(y_train_s1) * 100):.2f}%")
print(f"Test set fraud rate (Stage 1): {(y_test_s1.sum() / len(y_test_s1) * 100):.2f}%")

In [0]:
# Train Stage 1: Binary fraud detection classifier
print("\n" + "="*80)
print("STAGE 1: FRAUD DETECTION (BINARY CLASSIFIER)")
print("="*80 + "\n")

# Calculate scale_pos_weight for class imbalance
scale_pos_weight = (y_train_s1 == 0).sum() / (y_train_s1 == 1).sum()
print(f"Class imbalance ratio: {scale_pos_weight:.2f}")
print(f"Using scale_pos_weight={scale_pos_weight:.2f} to handle imbalance\n")

# Define XGBoost parameters
params_stage1 = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'tree_method': 'hist',
    'use_label_encoder': False
}

print("Model parameters (Stage 1):")
for key, value in params_stage1.items():
    print(f"  {key}: {value}")

# Train model
print("\nTraining Stage 1 model...")
import time
start_time = time.time()

model_stage1 = xgb.XGBClassifier(**params_stage1)
model_stage1.fit(
    X_train, y_train_s1,
    eval_set=[(X_train, y_train_s1), (X_test, y_test_s1)],
    verbose=False
)

training_time_s1 = time.time() - start_time
print(f"✓ Stage 1 model training completed in {training_time_s1:.2f} seconds")

In [0]:
# Evaluate Stage 1: Fraud Detection Model
print("\n" + "="*80)
print("STAGE 1 EVALUATION: FRAUD DETECTION")
print("="*80 + "\n")

print("Making predictions on test set...")
y_pred_s1 = model_stage1.predict(X_test)
y_pred_proba_s1 = model_stage1.predict_proba(X_test)[:, 1]
print("✓ Predictions completed\n")

# Calculate metrics
accuracy_s1 = accuracy_score(y_test_s1, y_pred_s1)
precision_s1 = precision_score(y_test_s1, y_pred_s1, zero_division=0)
recall_s1 = recall_score(y_test_s1, y_pred_s1)
f1_s1 = f1_score(y_test_s1, y_pred_s1)
roc_auc_s1 = roc_auc_score(y_test_s1, y_pred_proba_s1)

print("Stage 1 Performance Metrics:")
print(f"  Accuracy:  {accuracy_s1:.4f} ({accuracy_s1*100:.2f}%)")
print(f"  Precision: {precision_s1:.4f} ({precision_s1*100:.2f}%)")
print(f"  Recall:    {recall_s1:.4f} ({recall_s1*100:.2f}%)")
print(f"  F1 Score:  {f1_s1:.4f}")
print(f"  ROC AUC:   {roc_auc_s1:.4f}")

# Classification report
print("\n" + "-"*80)
print("Classification Report (Stage 1):")
print("-"*80)
print(classification_report(y_test_s1, y_pred_s1, target_names=['Legitimate', 'Fraud'], zero_division=0))

# Confusion matrix
print("\n" + "-"*80)
print("Confusion Matrix (Stage 1):")
print("-"*80)
cm_s1 = confusion_matrix(y_test_s1, y_pred_s1)
print(cm_s1)
print(f"\nTrue Negatives:  {cm_s1[0,0]:,}")
print(f"False Positives: {cm_s1[0,1]:,}")
print(f"False Negatives: {cm_s1[1,0]:,}")
print(f"True Positives:  {cm_s1[1,1]:,}")

print(f"\n✓ Stage 1 (Fraud Detection) complete: {roc_auc_s1:.4f} ROC AUC")

In [0]:
# Train Stage 2: Pattern Type Classifier (Multiclass)
print("\n" + "="*80)
print("STAGE 2: PATTERN TYPE CLASSIFICATION (MULTICLASS)")
print("="*80 + "\n")

print("Purpose: Classify WHAT TYPE of suspicious pattern a transaction exhibits")
print("Classes: 0=Legitimate, 1=Amount Anomaly, 2=Temporal Anomaly, 3=Merchant Fraud, 4=High-Risk Pattern\n")

# Train multiclass classifier for pattern types
params_stage2 = {
    'objective': 'multi:softmax',
    'num_class': 5,  # 0-4 pattern types
    'eval_metric': 'mlogloss',
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'tree_method': 'hist',
    'use_label_encoder': False
}

print("Model parameters (Stage 2):")
for key, value in params_stage2.items():
    print(f"  {key}: {value}")

print("\nTraining Stage 2 model...")
start_time_s2 = time.time()

model_stage2 = xgb.XGBClassifier(**params_stage2)
model_stage2.fit(
    X_train, y_train_s2,
    eval_set=[(X_train, y_train_s2), (X_test, y_test_s2)],
    verbose=False
)

training_time_s2 = time.time() - start_time_s2
print(f"✓ Stage 2 model training completed in {training_time_s2:.2f} seconds")

# Evaluate Stage 2
print("\n" + "-"*80)
print("STAGE 2 EVALUATION: PATTERN TYPE CLASSIFICATION")
print("-"*80 + "\n")

y_pred_s2 = model_stage2.predict(X_test)
y_pred_proba_s2 = model_stage2.predict_proba(X_test)

accuracy_s2 = accuracy_score(y_test_s2, y_pred_s2)

print("Stage 2 Performance Metrics:")
print(f"  Accuracy: {accuracy_s2:.4f} ({accuracy_s2*100:.2f}%)\n")

# Check which classes are present in test set
unique_classes = sorted(y_test_s2.unique())
pattern_names_all = ['Legitimate', 'Amount Anomaly', 'Temporal Anomaly', 'Merchant Fraud', 'High-Risk Pattern']
pattern_names_present = [pattern_names_all[i] for i in unique_classes]

print(f"Classes present in test set: {unique_classes}")
print(f"Pattern names: {pattern_names_present}\n")

# Classification report for pattern types
print("Classification Report (Stage 2 - Pattern Types):")
print("-"*80)
print(classification_report(y_test_s2, y_pred_s2, 
                            labels=unique_classes,
                            target_names=pattern_names_present, 
                            zero_division=0))

# Confusion matrix for Stage 2
cm_s2 = confusion_matrix(y_test_s2, y_pred_s2, labels=unique_classes)
print("\nConfusion Matrix (Stage 2):")
print(cm_s2)

print(f"\n✓ Stage 2 (Pattern Type Classification) complete: {accuracy_s2:.4f} accuracy")

In [0]:
# Visualize model performance
print("\n" + "="*80)
print("PERFORMANCE VISUALIZATIONS")
print("="*80 + "\n")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Confusion Matrix Heatmap
ax1 = axes[0, 0]
sns.heatmap(cm_s1, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
ax1.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax1.set_ylabel('Actual', fontsize=12)
ax1.set_xlabel('Predicted', fontsize=12)

# 2. ROC Curve
ax2 = axes[0, 1]
fpr, tpr, thresholds = roc_curve(y_test_s1, y_pred_proba_s1)
ax2.plot(fpr, tpr, linewidth=2, label=f'ROC (AUC = {roc_auc_s1:.3f})')
ax2.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate', fontsize=12)
ax2.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# 3. Precision-Recall Curve
ax3 = axes[1, 0]
precision_curve, recall_curve, _ = precision_recall_curve(y_test_s1, y_pred_proba_s1)
ax3.plot(recall_curve, precision_curve, linewidth=2, color='green')
ax3.set_xlabel('Recall', fontsize=12)
ax3.set_ylabel('Precision', fontsize=12)
ax3.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# 4. Feature Importance (Top 15)
ax4 = axes[1, 1]
feature_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_stage1.feature_importances_
}).sort_values('importance', ascending=False).head(15)

ax4.barh(range(len(feature_importance_df)), feature_importance_df['importance'], color='steelblue')
ax4.set_yticks(range(len(feature_importance_df)))
ax4.set_yticklabels(feature_importance_df['feature'], fontsize=9)
ax4.set_xlabel('Importance', fontsize=12)
ax4.set_title('Top 15 Feature Importance', fontsize=14, fontweight='bold')
ax4.invert_yaxis()

plt.tight_layout()
plt.show()

print("✓ Performance visualizations created")

In [0]:
# Feature importance analysis
print("\n" + "="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80 + "\n")

feature_importance_full = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_stage1.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features:")
for idx, row in feature_importance_full.head(20).iterrows():
    print(f"  {row['feature']:30s}  {row['importance']:.6f}")

print(f"\n✓ Total features: {len(feature_cols)}")
print(f"✓ Top feature: {feature_importance_full.iloc[0]['feature']} (importance: {feature_importance_full.iloc[0]['importance']:.4f})")

In [0]:
# SHAP explainability (optional - can be time consuming)
print("\n" + "="*80)
print("SHAP EXPLAINABILITY (ON SAMPLE)")
print("="*80 + "\n")

print("Calculating SHAP values on sample (1000 transactions)...")

# Use a small sample for SHAP (faster)
sample_size = min(1000, len(X_test))
X_sample = X_test.sample(n=sample_size, random_state=42)

explainer = shap.TreeExplainer(model_stage1)
shap_values = explainer.shap_values(X_sample)
print("✓ SHAP values calculated\n")

# SHAP summary plot
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=False, max_display=20)
plt.title('SHAP Feature Importance Summary (Top 20)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("✓ SHAP summary plot created")

In [0]:
# Save models with MLflow tracking
print("\n" + "="*80)
print("SAVING MODELS WITH MLFLOW TRACKING")
print("="*80 + "\n")

import pickle
import json
from datetime import datetime
import mlflow
import mlflow.xgboost
from mlflow.models import infer_signature

# Set MLflow experiment
username = spark.sql("SELECT current_user()").collect()[0][0]
experiment_name = f"/Users/{username}/suraksha_baseline"
mlflow.set_experiment(experiment_name)
print(f"✓ MLflow experiment set: {experiment_name}")

# Start MLflow run
with mlflow.start_run(run_name="baseline_two_stage_training") as run:
    # Log parameters
    mlflow.log_param("source_table", source_table)
    mlflow.log_param("feature_count", len(feature_cols))
    mlflow.log_param("scale_pos_weight_s1", scale_pos_weight_s1)
    mlflow.log_param("model_type", "XGBoost Two-Stage Classifier")
    mlflow.log_param("solution_type", "baseline")
    mlflow.log_param("fraud_types", "4_pattern_based")
    
    # Log Stage 1 metrics
    mlflow.log_metric("s1_accuracy", accuracy_s1)
    mlflow.log_metric("s1_precision", precision_s1)
    mlflow.log_metric("s1_recall", recall_s1)
    mlflow.log_metric("s1_f1", f1_s1)
    mlflow.log_metric("s1_roc_auc", roc_auc_s1)
    mlflow.log_metric("s1_training_time", training_time_s1)
    
    # Log Stage 2 metrics
    mlflow.log_metric("s2_accuracy", accuracy_s2)
    mlflow.log_metric("s2_f1_weighted", f1_s2)
    mlflow.log_metric("s2_training_time", training_time_s2)
    
    # Log per-class metrics for Stage 2
    for label, metrics in class_report_s2.items():
        if isinstance(metrics, dict) and 'f1-score' in metrics:
            mlflow.log_metric(f"s2_f1_class_{label}", metrics['f1-score'])
    
    # Create signature and log Stage 1 model
    signature_s1 = infer_signature(X_train, model_stage1.predict(X_train))
    mlflow.xgboost.log_model(
        xgb_model=model_stage1,
        artifact_path="stage1_fraud_detector",
        signature=signature_s1,
        registered_model_name="suraksha_baseline_stage1"
    )
    print("✓ Stage 1 model logged to MLflow")
    
    # Create signature and log Stage 2 model
    signature_s2 = infer_signature(X_train, model_stage2.predict(X_train))
    mlflow.xgboost.log_model(
        xgb_model=model_stage2,
        artifact_path="stage2_pattern_classifier",
        signature=signature_s2,
        registered_model_name="suraksha_baseline_stage2"
    )
    print("✓ Stage 2 model logged to MLflow")
    
    # Save feature names as artifact
    feature_names_artifact = {'features': feature_cols}
    mlflow.log_dict(feature_names_artifact, "feature_names.json")
    print("✓ Feature names logged")
    
    run_id = run.info.run_id
    print(f"\n✓ MLflow run ID: {run_id}")

# Also save to Delta table for backward compatibility
print("\nSaving to Delta table for backward compatibility...")

# Serialize both models into a single artifact
model_artifact = {
    'model_stage1': model_stage1,
    'model_stage2': model_stage2,
    'feature_names': feature_cols,
    'feature_count': len(feature_cols),
    's1_accuracy': accuracy_s1,
    's1_precision': precision_s1,
    's1_recall': recall_s1,
    's1_f1': f1_s1,
    's1_roc_auc': roc_auc_s1,
    's1_training_time': training_time_s1,
    's2_accuracy': accuracy_s2,
    's2_f1': f1_s2,
    's2_training_time': training_time_s2,
    'scale_pos_weight_s1': scale_pos_weight_s1,
    'trained_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model_type': 'XGBoost Two-Stage Classifier',
    'solution_type': 'baseline',
    'fraud_types': '4_pattern_based',
    'source_table': source_table,
    'mlflow_run_id': run_id
}

model_bytes = pickle.dumps(model_artifact)
model_size = len(model_bytes)
print(f"✓ Models serialized: {model_size / 1024:.2f} KB ({model_size / (1024*1024):.2f} MB)")

metadata = json.dumps({
    "feature_count": len(feature_cols),
    "s1_accuracy": float(accuracy_s1),
    "s1_precision": float(precision_s1),
    "s1_recall": float(recall_s1),
    "s1_f1": float(f1_s1),
    "s1_roc_auc": float(roc_auc_s1),
    "s2_accuracy": float(accuracy_s2),
    "s2_f1": float(f1_s2),
    "model_type": "XGBoost Two-Stage Classifier",
    "solution_type": "baseline",
    "fraud_types": "4_pattern_based",
    "mlflow_run_id": run_id
})

model_df = spark.createDataFrame([
    (
        'suraksha_baseline_v1',
        model_bytes,
        metadata,
        datetime.now()
    )
], ['model_name', 'model_binary', 'metadata', 'saved_at'])

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.models")
model_table = 'workspace.models.baseline_binary_classifier'
model_df.write.format('delta').mode('overwrite').saveAsTable(model_table)

print(f"✓ Models saved to Delta table: {model_table}")

print("\n" + "="*80)
print("✓ MODELS SAVED SUCCESSFULLY")
print("="*80)
print(f"\nMLflow Models:")
print(f"  - models:/suraksha_baseline_stage1/latest")
print(f"  - models:/suraksha_baseline_stage2/latest")
print(f"\nDelta Table: {model_table}")
print(f"\nArtifact Contents:")
print(f"  - Stage 1: Binary fraud detector (accuracy={accuracy_s1:.4f}, F1={f1_s1:.4f}, ROC AUC={roc_auc_s1:.4f})")
print(f"  - Stage 2: Pattern classifier (accuracy={accuracy_s2:.4f}, F1={f1_s2:.4f})")
print(f"  - {len(feature_cols)} features")
print(f"  - MLflow run: {run_id}")

In [0]:
# Test model inference
print("\n" + "="*80)
print("MODEL INFERENCE TEST")
print("="*80 + "\n")

# Load the saved model from Delta table
print("Loading saved model from Delta table...")
import pickle

model_table = 'workspace.models.baseline_binary_classifier'
model_row = spark.read.table(model_table).collect()[0]

# Deserialize model artifact
loaded_artifact = pickle.loads(model_row['model_binary'])
loaded_model = loaded_artifact['model_stage1']
loaded_features = loaded_artifact['feature_names']

print(f"✓ Model loaded successfully")
print(f"✓ Model name: {model_row['model_name']}")
print(f"✓ Features: {len(loaded_features)}")
print(f"✓ Training metrics: Accuracy={loaded_artifact['s1_accuracy']:.4f}, F1={loaded_artifact['s1_f1']:.4f}")
print(f"✓ Saved at: {model_row['saved_at']}")

# Test on 10 samples
test_samples = X_test.head(10)
predictions = loaded_model.predict(test_samples)
predictions_proba = loaded_model.predict_proba(test_samples)

print("\nTest predictions (first 10 samples):")
test_results = pd.DataFrame({
    'actual': y_test_s1.head(10).values,
    'predicted': predictions,
    'fraud_prob': [f"{p:.4f}" for p in predictions_proba[:, 1]],
    'legit_prob': [f"{p:.4f}" for p in predictions_proba[:, 0]]
})
print(test_results.to_string(index=False))

print("\n✓ Model inference test successful")
print(f"✓ All predictions match actual labels: {(predictions == y_test_s1.head(10).values).all()}")

## ✓ Binary Classification Training Summary

### Model Details
* **Model Name**: `suraksha_baseline_v1`
* **Algorithm**: XGBoost Binary Classifier
* **Solution Type**: Baseline (Pattern-based proof-of-concept)
* **Features**: 25 pattern-based features (no user tracking)
* **Target**: Pattern-based synthetic labels (12,489 fraud cases, 5%)
* **Storage**: Delta table at `workspace.models.baseline_binary_classifier`

### Training Results
* Model trained on 200K transactions, tested on 50K
* Class balance: 95% legitimate, 5% fraud (pattern-based)
* High performance on pattern-defined fraud (expected with synthetic labels)
* Feature importance and SHAP analysis show pattern features work effectively
* Model saved to Delta table for serverless inference

### Pattern-Based Fraud Detection (4 Types)
1. **Amount Anomaly** (5,347 cases) - Statistical outliers (z-score > 3, percentile > 95)
2. **Temporal Anomaly** (6,305 cases) - Odd hours (2-5 AM), late night
3. **Merchant Fraud** (1 case) - High-risk merchant + high amount
4. **High-Risk Pattern** (836 cases) - Multiple indicators (risk score ≥ 8)

### What This Baseline Demonstrates

✓ **Feature Engineering Works**: Pattern features effectively detect fraud when definitions align  
✓ **Transaction-Level Detection**: No user tracking required for pattern-based fraud  
✓ **Adaptable to Constraints**: Works with official dataset structure (17 columns)  
✓ **Databricks Integration**: Delta Lake, Spark, Unity Catalog, serverless compute  

❌ **Limitations** (Addressed by Advanced Solution):  
❌ **No Behavioral Fraud**: Cannot detect velocity attacks, mule accounts, SIM swap  
❌ **Synthetic Labels**: Official dataset fraud labels don't match pattern assumptions  
❌ **Binary Only**: Cannot distinguish between fraud types for targeted responses  
❌ **Limited Accuracy on Real Fraud**: Pattern features alone insufficient for unknown fraud definitions  

### Top Features (by Importance)
1. **total_risk_score** - Composite risk indicator (temporal + amount + pattern)
2. **is_high_amount** - High-value transaction flag (>= 95th percentile)
3. **amount_zscore** - Statistical deviation from mean

### Baseline vs Advanced Solution

| Aspect | Baseline (This Notebook) | Advanced |
|--------|----------|----------|
| **Data Source** | Official dataset (250K) | Synthetic with user IDs (120K) |
| **User Tracking** | ❌ None | ✓ VPAs, Device IDs, SIM data |
| **Fraud Types** | 4 pattern-based | 9 (5 behavior + 4 pattern) |
| **Classification** | Binary | Multiclass (9 classes) |
| **Labels** | Pattern-based (synthetic) | Behavior-based (realistic) |
| **Fraud Rate** | 5% (pattern-defined) | 5% (multiple types) |
| **Performance** | High on patterns | Realistic on behavioral fraud |
| **Production Ready** | ❌ Proof-of-concept | ✓ Yes |

### Model Storage (Judge-Reproducible)
* **Method**: Delta table (no MLflow dependencies)
* **Table**: `workspace.models.baseline_binary_classifier`
* **Contents**: Serialized XGBoost model, feature names, metadata, metrics
* **Size**: ~125 KB (lightweight, fast loading)
* **Reproducibility**: All paths use `current_user()`, no hard-coded usernames

### Next Steps

1. ▶ **Review 04_model_serving** for inference pipeline
2. ▶ **Compare with advanced solution** (behavioral fraud detection)
3. ▶ **Connect to Streamlit app** (dual-mode: baseline + advanced)
4. ▶ **Integrate RAG** for fraud explanation (RBI guidelines)
5. ▶ **Test IndicTrans2** for Hindi translation (Indian model bonus)

### For Judges: Understanding This Approach

#### Why Pattern-Based Synthetic Labels?

**Official Dataset Challenge:**
* Official fraud labels (`is_fraud`): 480 cases (0.19%)
* Our pattern features: temporal, amount, merchant risk
* **Correlation analysis**: ROC AUC 0.47 (worse than random!)
* Conclusion: Official dataset uses unknown fraud criteria

**Solution: Proof-of-Concept with Patterns:**
* Create synthetic labels based on pattern assumptions
* Demonstrates feature engineering effectiveness
* Shows Databricks platform integration (Delta, Spark, Unity Catalog)
* **Advanced solution** addresses real behavioral fraud

**What Judges Get:**
* **Baseline**: Pattern features work when definitions align (this notebook)
* **Advanced**: Behavioral features detect real fraud types (velocity, mule accounts)
* **Complete Spectrum**: Transaction-level → User-level fraud detection

#### Reproducibility Steps
1. Run `01_data_ingestion` → Load official dataset to bronze layer
2. Run `02_pattern_features` → Engineer 25 pattern-based features
3. Run `03_binary_training` (this notebook) → Train XGBoost on pattern labels
4. Model saved to Delta table, ready for inference

All paths use `spark.sql("SELECT current_user()")` - judges' username auto-detected!